# MeetingScheduler: 회의 일정 자동 조율 Agent

## 개요
이 노트북은 **회의 참가자들의 일정을 자동으로 수집·분석하여 최적의 회의 시간을 추천하고 확정**하는 LangGraph 기반 Agent입니다.

### 주요 기능
- 👥 **참가자 일정 통합 수집**: Teams, 사내 시스템, 메신저에서 자동 수집
- 📌 **불변 일정 식별**: 고객사 미팅 등 변경 불가 일정 태깅
- ⏰ **교집합 시간대 탐색**: 모든 참가자가 가능한 시간 찾기
- 🔄 **대체 시간 추천**: 공통 시간 없을 때 우선순위 반영
- 📧 **자동 알림**: Teams/메신저를 통한 초대장 발송
- 🔁 **폴백 및 재시도**: API 실패 시 자동 복구
- 💾 **상태 체크포인트**: 단계별 결과 저장 및 복구

### 워크플로우
```
입력 → 수집 → 정리 → 태깅 → 교집합 판단 → 대체안/확정 → 알림 → 완료
```

### 고급 요소
1. **조건부 재시도/폴백**: API 실패 시 임계값 재시도 후 규칙 기반 대체
2. **상태 체크포인트**: 실패 시 마지막 성공 지점부터 재개
3. **최적 회의 시간 추출**: 우선순위 + 중요도 + 선호도 가중합


## Section 1: 환경 설정 및 데이터 스키마 정의

필요한 라이브러리 임포트 및 Pydantic 스키마 정의

In [11]:
# 1. 필수 라이브러리 임포트
import sys
sys.path.append('.')

import os
from dotenv import load_dotenv
load_dotenv()

from datetime import datetime, timedelta, time
from typing import Optional, Dict, List, Tuple
from pydantic import BaseModel, Field
from enum import Enum
import json
import pandas as pd
from collections import defaultdict
import logging
from functools import wraps

# LangGraph 관련
from langgraph.graph import StateGraph, START, END
from langgraph.types import StateSnapshot

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ 라이브러리 임포트 완료")
print(f"  - pandas: {pd.__version__}")
print("  - LangGraph: 설치됨")

✓ 라이브러리 임포트 완료
  - pandas: 3.0.2
  - LangGraph: 설치됨


In [12]:
# 2. 데이터 스키마 정의
class Priority(str, Enum):
    """참가자 우선순위"""
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"


class Participant(BaseModel):
    """회의 참가자 정보"""
    name: str = Field(..., description="참가자 이름")
    email: str = Field(..., description="참가자 이메일")
    department: str = Field(..., description="부서")
    priority: Priority = Field(default=Priority.MEDIUM, description="우선순위")


class CalendarEvent(BaseModel):
    """캘린더 이벤트"""
    title: str
    start_time: datetime
    end_time: datetime
    is_fixed: bool = False  # 변경 불가능 여부
    is_external: bool = False  # 외부 일정 여부
    participant: str


class TimeSlot(BaseModel):
    """가용 시간대"""
    start_time: datetime
    end_time: datetime
    available_participants: List[str]


class MeetingRecommendation(BaseModel):
    """회의 추천"""
    recommended_time: datetime
    available_participants: List[str]
    excluded_participants: List[str] = []
    confidence_score: float
    reason: str


class AgentState(BaseModel):
    """워크플로우 상태"""
    step: str = "input"
    participants: List[Participant] = []
    required_duration_minutes: int = 60
    preferred_date_range: Optional[Tuple[datetime, datetime]] = None
    calendar_data: Dict[str, List[CalendarEvent]] = {}
    fixed_schedules: List[CalendarEvent] = []
    common_available_slots: List[TimeSlot] = []
    has_common_slot: bool = False
    final_recommendation: Optional[MeetingRecommendation] = None
    checkpoint_data: Dict = {}
    error_log: List[str] = []
    retry_count: int = 0


print("✓ Pydantic 스키마 정의 완료")

✓ Pydantic 스키마 정의 완료


## Section 2: 참가자/일정 더미 데이터 및 커넥터 인터페이스

In [13]:
# 3. 샘플 참가자 데이터
SAMPLE_PARTICIPANTS = [
    Participant(name="Alice", email="alice@company.com", department="Sales", priority=Priority.HIGH),
    Participant(name="Bob", email="bob@company.com", department="Engineering", priority=Priority.HIGH),
    Participant(name="Charlie", email="charlie@company.com", department="Marketing", priority=Priority.MEDIUM),
    Participant(name="Diana", email="diana@company.com", department="Operations", priority=Priority.MEDIUM),
    Participant(name="Eve", email="eve@company.com", department="Executive", priority=Priority.HIGH),
]

# 4. 더미 캘린더 데이터 생성
def generate_sample_calendar_data() -> Dict[str, List[CalendarEvent]]:
    """테스트용 샘플 캘린더 데이터"""
    today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    base_date = today + timedelta(days=1)  # 내일부터
    
    calendar_data = {
        "Alice": [
            CalendarEvent(
                title="고객 미팅 (고정)",
                start_time=base_date + timedelta(hours=10),
                end_time=base_date + timedelta(hours=11),
                is_fixed=True,
                is_external=True,
                participant="Alice"
            ),
            CalendarEvent(
                title="팀 스탠드업",
                start_time=base_date + timedelta(hours=9),
                end_time=base_date + timedelta(hours=9, minutes=30),
                participant="Alice"
            ),
        ],
        "Bob": [
            CalendarEvent(
                title="프로젝트 리뷰",
                start_time=base_date + timedelta(hours=14),
                end_time=base_date + timedelta(hours=15),
                participant="Bob"
            ),
        ],
        "Charlie": [
            CalendarEvent(
                title="마케팅 회의",
                start_time=base_date + timedelta(hours=15),
                end_time=base_date + timedelta(hours=16, minutes=30),
                participant="Charlie"
            ),
        ],
        "Diana": [
            CalendarEvent(
                title="운영 회의",
                start_time=base_date + timedelta(hours=11),
                end_time=base_date + timedelta(hours=12),
                participant="Diana"
            ),
        ],
        "Eve": [
            CalendarEvent(
                title="경영진 회의",
                start_time=base_date + timedelta(hours=9),
                end_time=base_date + timedelta(hours=10),
                is_fixed=True,
                participant="Eve"
            ),
        ],
    }
    return calendar_data

# 5. 커넥터 인터페이스
from abc import ABC, abstractmethod

class CalendarConnector(ABC):
    """캘린더 데이터 수집 커넥터 추상 클래스"""
    
    def __init__(self, name: str):
        self.name = name
    
    @abstractmethod
    def fetch_calendar(self, participant: Participant) -> List[CalendarEvent]:
        """참가자의 캘린더 데이터 수집"""
        pass
    
    def __repr__(self):
        return f"{self.name}Connector"


class TeamsConnector(CalendarConnector):
    """Teams API 커넥터"""
    def fetch_calendar(self, participant: Participant) -> List[CalendarEvent]:
        logger.info(f"[Teams] 수집 중: {participant.name}")
        sample_data = generate_sample_calendar_data()
        return sample_data.get(participant.name, [])


class InternalDBConnector(CalendarConnector):
    """사내 DB 커넥터"""
    def fetch_calendar(self, participant: Participant) -> List[CalendarEvent]:
        logger.info(f"[Internal DB] 수집 중: {participant.name}")
        sample_data = generate_sample_calendar_data()
        return sample_data.get(participant.name, [])


class MessengerConnector(CalendarConnector):
    """메신저 연동 커넥터"""
    def fetch_calendar(self, participant: Participant) -> List[CalendarEvent]:
        logger.info(f"[Messenger] 수집 중: {participant.name}")
        return []  # 메신저에서는 일정 없음

# 커넥터 인스턴스
connectors = {
    "teams": TeamsConnector("Teams"),
    "internal_db": InternalDBConnector("Internal DB"),
    "messenger": MessengerConnector("Messenger"),
}

print("✓ 샘플 데이터 및 커넥터 구현 완료")
print(f"  - 참가자: {len(SAMPLE_PARTICIPANTS)}명")
print(f"  - 커넥터: {', '.join(connectors.keys())}")

✓ 샘플 데이터 및 커넥터 구현 완료
  - 참가자: 5명
  - 커넥터: teams, internal_db, messenger


## Section 3: 사내 시스템 데이터 수집 파이프라인

In [14]:
# 6. 다중 소스 데이터 수집 파이프라인
def fetch_calendar_from_multiple_sources(
    participant: Participant,
    connectors_list: List[CalendarConnector],
    simulate_failure: bool = False,
    failure_rate: float = 0.0
) -> List[CalendarEvent]:
    """
    여러 소스에서 참가자 일정 수집
    
    Args:
        participant: 대상 참가자
        connectors_list: 사용할 커넥터 리스트
        simulate_failure: 실패 시뮬레이션 여부
        failure_rate: 실패율 (0-1)
    """
    all_events = []
    import random
    
    for connector in connectors_list:
        try:
            # 실패 시뮬레이션
            if simulate_failure and random.random() < failure_rate:
                raise Exception(f"{connector.name} API 오류")
            
            events = connector.fetch_calendar(participant)
            all_events.extend(events)
        except Exception as e:
            logger.warning(f"{connector.name}에서 {participant.name} 일정 수집 실패: {str(e)}")
            continue
    
    return all_events


def collect_all_participants_calendar(
    participants: List[Participant],
    simulate_failure: bool = False
) -> Dict[str, List[CalendarEvent]]:
    """모든 참가자의 일정 수집"""
    calendar_data = {}
    connectors_list = list(connectors.values())
    
    for participant in participants:
        events = fetch_calendar_from_multiple_sources(
            participant,
            connectors_list,
            simulate_failure=simulate_failure
        )
        calendar_data[participant.name] = events
    
    return calendar_data


# 테스트: 정상 수집
calendar_sample = collect_all_participants_calendar(SAMPLE_PARTICIPANTS)
print(f"✓ 캘린더 데이터 수집 완료")
print(f"  - 참가자별 일정 수")
for name, events in calendar_sample.items():
    print(f"    {name}: {len(events)}건")


# 테스트: 실패 시뮬레이션
print("\n⚠️ 실패 시뮬레이션 테스트")
calendar_with_failure = collect_all_participants_calendar(
    SAMPLE_PARTICIPANTS,
    simulate_failure=True
)
print(f"✓ 일부 실패 후에도 수집 완료")

# DataFrame으로 병합
def calendar_to_dataframe(calendar_data: Dict[str, List[CalendarEvent]]) -> pd.DataFrame:
    """캘린더 데이터를 DataFrame으로 변환"""
    rows = []
    for participant_name, events in calendar_data.items():
        for event in events:
            rows.append({
                'participant': participant_name,
                'title': event.title,
                'start_time': event.start_time,
                'end_time': event.end_time,
                'is_fixed': event.is_fixed,
                'is_external': event.is_external,
            })
    return pd.DataFrame(rows)

calendar_df = calendar_to_dataframe(calendar_sample)
print("\n캘린더 데이터 (DataFrame):")
print(calendar_df.head(10))

INFO:__main__:[Teams] 수집 중: Alice
INFO:__main__:[Internal DB] 수집 중: Alice
INFO:__main__:[Messenger] 수집 중: Alice
INFO:__main__:[Teams] 수집 중: Bob
INFO:__main__:[Internal DB] 수집 중: Bob
INFO:__main__:[Messenger] 수집 중: Bob
INFO:__main__:[Teams] 수집 중: Charlie
INFO:__main__:[Internal DB] 수집 중: Charlie
INFO:__main__:[Messenger] 수집 중: Charlie
INFO:__main__:[Teams] 수집 중: Diana
INFO:__main__:[Internal DB] 수집 중: Diana
INFO:__main__:[Messenger] 수집 중: Diana
INFO:__main__:[Teams] 수집 중: Eve
INFO:__main__:[Internal DB] 수집 중: Eve
INFO:__main__:[Messenger] 수집 중: Eve
INFO:__main__:[Teams] 수집 중: Alice
INFO:__main__:[Internal DB] 수집 중: Alice
INFO:__main__:[Messenger] 수집 중: Alice
INFO:__main__:[Teams] 수집 중: Bob
INFO:__main__:[Internal DB] 수집 중: Bob
INFO:__main__:[Messenger] 수집 중: Bob
INFO:__main__:[Teams] 수집 중: Charlie
INFO:__main__:[Internal DB] 수집 중: Charlie
INFO:__main__:[Messenger] 수집 중: Charlie
INFO:__main__:[Teams] 수집 중: Diana
INFO:__main__:[Internal DB] 수집 중: Diana
INFO:__main__:[Messenger] 수집 중: Dian

✓ 캘린더 데이터 수집 완료
  - 참가자별 일정 수
    Alice: 4건
    Bob: 2건
    Charlie: 2건
    Diana: 2건
    Eve: 2건

⚠️ 실패 시뮬레이션 테스트
✓ 일부 실패 후에도 수집 완료

캘린더 데이터 (DataFrame):
  participant       title          start_time            end_time  is_fixed  \
0       Alice  고객 미팅 (고정) 2026-04-17 10:00:00 2026-04-17 11:00:00      True   
1       Alice      팀 스탠드업 2026-04-17 09:00:00 2026-04-17 09:30:00     False   
2       Alice  고객 미팅 (고정) 2026-04-17 10:00:00 2026-04-17 11:00:00      True   
3       Alice      팀 스탠드업 2026-04-17 09:00:00 2026-04-17 09:30:00     False   
4         Bob     프로젝트 리뷰 2026-04-17 14:00:00 2026-04-17 15:00:00     False   
5         Bob     프로젝트 리뷰 2026-04-17 14:00:00 2026-04-17 15:00:00     False   
6     Charlie      마케팅 회의 2026-04-17 15:00:00 2026-04-17 16:30:00     False   
7     Charlie      마케팅 회의 2026-04-17 15:00:00 2026-04-17 16:30:00     False   
8       Diana       운영 회의 2026-04-17 11:00:00 2026-04-17 12:00:00     False   
9       Diana       운영 회의 2026-04-17 11:00:00 2026-04-1

## Section 4: 일정 표준화 및 타임존 정규화

In [15]:
# 7. 일정 표준화 및 정규화
def standardize_calendar_data(
    calendar_data: Dict[str, List[CalendarEvent]]
) -> Dict[str, List[CalendarEvent]]:
    """
    원시 날짜 표준화, 중복 제거, 결측치 보정
    """
    standardized = {}
    
    for participant_name, events in calendar_data.items():
        # 시간순 정렬
        sorted_events = sorted(events, key=lambda e: e.start_time)
        
        # 중복 제거 (같은 시간대, 같은 제목)
        unique_events = []
        seen = set()
        
        for event in sorted_events:
            key = (event.start_time, event.end_time, event.title, event.participant)
            if key not in seen:
                seen.add(key)
                unique_events.append(event)
        
        # 겹치는 이벤트 병합
        merged_events = []
        for event in unique_events:
            if merged_events and merged_events[-1].end_time >= event.start_time:
                # 이전 이벤트와 겹침 -> 병합
                last = merged_events[-1]
                merged_event = CalendarEvent(
                    title=f"{last.title} + {event.title}",
                    start_time=last.start_time,
                    end_time=max(last.end_time, event.end_time),
                    is_fixed=last.is_fixed or event.is_fixed,
                    is_external=last.is_external or event.is_external,
                    participant=participant_name
                )
                merged_events[-1] = merged_event
            else:
                merged_events.append(event)
        
        standardized[participant_name] = merged_events
    
    return standardized


# 표준화 실행
standardized_calendar = standardize_calendar_data(calendar_sample)
print("✓ 일정 표준화 완료")
print(f"  - 표준화 후 일정 수")
for name, events in standardized_calendar.items():
    print(f"    {name}: {len(events)}건 (변경 불가: {sum(1 for e in events if e.is_fixed)}건)")


# 타임존 정규화 (한국 표준시)
from zoneinfo import ZoneInfo

KST = ZoneInfo("Asia/Seoul")

def normalize_timezone(calendar_data: Dict[str, List[CalendarEvent]]) -> Dict[str, List[CalendarEvent]]:
    """모든 시간을 KST로 정규화"""
    normalized = {}
    
    for participant_name, events in calendar_data.items():
        normalized_events = []
        for event in events:
            # naive datetime을 KST로 인식
            if event.start_time.tzinfo is None:
                start = event.start_time.replace(tzinfo=KST)
            else:
                start = event.start_time.astimezone(KST)
            
            if event.end_time.tzinfo is None:
                end = event.end_time.replace(tzinfo=KST)
            else:
                end = event.end_time.astimezone(KST)
            
            normalized_event = CalendarEvent(
                title=event.title,
                start_time=start,
                end_time=end,
                is_fixed=event.is_fixed,
                is_external=event.is_external,
                participant=participant_name
            )
            normalized_events.append(normalized_event)
        
        normalized[participant_name] = normalized_events
    
    return normalized


# 타임존 정규화 실행
normalized_calendar = normalize_timezone(standardized_calendar)
print("\n✓ 타임존 정규화 완료 (KST)")

# 최종 데이터 확인
final_df = calendar_to_dataframe(normalized_calendar)
print("\n최종 캘린더 데이터:")
print(final_df.to_string(index=False))

✓ 일정 표준화 완료
  - 표준화 후 일정 수
    Alice: 2건 (변경 불가: 1건)
    Bob: 1건 (변경 불가: 0건)
    Charlie: 1건 (변경 불가: 0건)
    Diana: 1건 (변경 불가: 0건)
    Eve: 1건 (변경 불가: 1건)

✓ 타임존 정규화 완료 (KST)

최종 캘린더 데이터:
participant      title                start_time                  end_time  is_fixed  is_external
      Alice     팀 스탠드업 2026-04-17 09:00:00+09:00 2026-04-17 09:30:00+09:00     False        False
      Alice 고객 미팅 (고정) 2026-04-17 10:00:00+09:00 2026-04-17 11:00:00+09:00      True         True
        Bob    프로젝트 리뷰 2026-04-17 14:00:00+09:00 2026-04-17 15:00:00+09:00     False        False
    Charlie     마케팅 회의 2026-04-17 15:00:00+09:00 2026-04-17 16:30:00+09:00     False        False
      Diana      운영 회의 2026-04-17 11:00:00+09:00 2026-04-17 12:00:00+09:00     False        False
        Eve     경영진 회의 2026-04-17 09:00:00+09:00 2026-04-17 10:00:00+09:00      True        False


## Section 5: 불변 일정(고객사 등) 태깅 모델/규칙 구현

In [16]:
# 8. 불변 일정 식별 (LLM 기반 또는 규칙 기반)
# 키워드 기반 불변 일정 식별
FIXED_KEYWORDS = [
    "고객", "customer", "client", "external",
    "고정", "fixed", "immutable",
    "중요", "critical", "essential",
    "데드라인", "deadline",
]

def identify_fixed_schedules_rule_based(
    calendar_data: Dict[str, List[CalendarEvent]]
) -> List[CalendarEvent]:
    """
    규칙 기반 불변 일정 식별
    키워드 매칭 또는 is_fixed 플래그 사용
    """
    fixed_schedules = []
    
    for participant_name, events in calendar_data.items():
        for event in events:
            # 이미 is_fixed 플래그가 설정된 경우
            if event.is_fixed or event.is_external:
                fixed_schedules.append(event)
            else:
                # 제목에서 키워드 확인
                title_lower = event.title.lower()
                for keyword in FIXED_KEYWORDS:
                    if keyword.lower() in title_lower:
                        event.is_fixed = True
                        fixed_schedules.append(event)
                        break
    
    return fixed_schedules


# 불변 일정 식별
fixed_schedules = identify_fixed_schedules_rule_based(normalized_calendar)
print("✓ 불변 일정 식별 완료")
print(f"  - 총 불변 일정: {len(fixed_schedules)}건")
for event in fixed_schedules:
    print(f"    [{event.participant}] {event.title} ({event.start_time.strftime('%Y-%m-%d %H:%M')})")


# 신뢰도 점수 계산
class FixedScheduleScore(BaseModel):
    """불변 일정 신뢰도 점수"""
    event: CalendarEvent
    confidence_score: float
    reason: str


def calculate_fixed_schedule_confidence(
    calendar_data: Dict[str, List[CalendarEvent]]
) -> List[FixedScheduleScore]:
    """불변 일정 신뢰도 점수 계산"""
    scores = []
    
    for participant_name, events in calendar_data.items():
        for event in events:
            confidence = 0.0
            reason_parts = []
            
            # 기준 1: is_fixed 플래그 (100점)
            if event.is_fixed:
                confidence += 0.5
                reason_parts.append("is_fixed=True")
            
            # 기준 2: is_external (40점)
            if event.is_external:
                confidence += 0.4
                reason_parts.append("외부 일정")
            
            # 기준 3: 키워드 매칭
            title_lower = event.title.lower()
            for keyword in FIXED_KEYWORDS:
                if keyword.lower() in title_lower:
                    confidence += 0.1
                    reason_parts.append(f"키워드: {keyword}")
                    break
            
            # 기준 4: 참가자 우선순위
            if participant_name in ["Alice", "Eve"]:  # HIGH priority
                confidence += 0.05
                reason_parts.append("HIGH priority participant")
            
            scores.append(FixedScheduleScore(
                event=event,
                confidence_score=min(confidence, 1.0),
                reason=" | ".join(reason_parts) if reason_parts else "일반 일정"
            ))
    
    return scores


# 신뢰도 점수 계산
confidence_scores = calculate_fixed_schedule_confidence(normalized_calendar)
print("\n불변 일정 신뢰도 점수:")
for score in confidence_scores:
    if score.confidence_score > 0.5:
        print(f"  [{score.event.participant:10}] {score.event.title:30} | 신뢰도: {score.confidence_score:.2f} | {score.reason}")

✓ 불변 일정 식별 완료
  - 총 불변 일정: 2건
    [Alice] 고객 미팅 (고정) (2026-04-17 10:00)
    [Eve] 경영진 회의 (2026-04-17 09:00)

불변 일정 신뢰도 점수:
  [Alice     ] 고객 미팅 (고정)                     | 신뢰도: 1.00 | is_fixed=True | 외부 일정 | 키워드: 고객 | HIGH priority participant
  [Eve       ] 경영진 회의                         | 신뢰도: 0.55 | is_fixed=True | HIGH priority participant


## Section 6: 개인별 가용 시간대 계산 엔진

In [17]:
# 9. 개인별 가용 시간대 계산
WORK_START_HOUR = 9
WORK_END_HOUR = 18
LUNCH_START = 12
LUNCH_END = 13
SLOT_DURATION_MINUTES = 30


def calculate_available_slots(
    participant_name: str,
    events: List[CalendarEvent],
    date_range: Tuple[datetime, datetime],
    duration_minutes: int = 60
) -> List[TimeSlot]:
    """
    개인별 가용 시간대 계산 (30분 단위 슬롯)
    
    Args:
        participant_name: 참가자 이름
        events: 해당 참가자의 일정 목록
        date_range: 검색 날짜 범위 (start, end)
        duration_minutes: 필요 회의 시간
    
    Returns:
        TimeSlot 리스트
    """
    available_slots = []
    
    # 업무 시간 범위으로 시간대 생성
    current_date = date_range[0].date()
    end_date = date_range[1].date()
    
    while current_date <= end_date:
        # 점심 시간 제외한 업무 시간대
        day_start = datetime.combine(current_date, time(WORK_START_HOUR, 0), tzinfo=KST)
        day_end = datetime.combine(current_date, time(WORK_END_HOUR, 0), tzinfo=KST)
        
        current_time = day_start
        
        while current_time + timedelta(minutes=duration_minutes) <= day_end:
            slot_end = current_time + timedelta(minutes=duration_minutes)
            
            # 점심 시간 제외
            if LUNCH_START <= current_time.hour < LUNCH_END or LUNCH_START <= slot_end.hour < LUNCH_END:
                current_time += timedelta(minutes=SLOT_DURATION_MINUTES)
                continue
            
            # 기존 일정과 겹치는지 확인
            has_conflict = False
            for event in events:
                if event.start_time < slot_end and event.end_time > current_time:
                    has_conflict = True
                    break
            
            if not has_conflict:
                available_slots.append(TimeSlot(
                    start_time=current_time,
                    end_time=slot_end,
                    available_participants=[participant_name]
                ))
            
            current_time += timedelta(minutes=SLOT_DURATION_MINUTES)
        
        current_date += timedelta(days=1)
    
    return available_slots


# 각 참가자의 가용 시간대 계산
today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0, tzinfo=KST)
date_range = (today, today + timedelta(days=4))  # 5일간

available_slots_by_participant = {}
for participant_name, events in normalized_calendar.items():
    slots = calculate_available_slots(participant_name, events, date_range)
    available_slots_by_participant[participant_name] = slots

print("✓ 개인별 가용 시간대 계산 완료")
print(f"  - 검색 기간: {date_range[0].date()} ~ {date_range[1].date()}")
for name, slots in available_slots_by_participant.items():
    print(f"    {name}: {len(slots):3}개 슬롯 ({len(slots) * SLOT_DURATION_MINUTES}분)")

# 가용 시간대 시각화
print("\n개인별 가용 시간대 (샘플 - 첫 2일):")
for participant_name, slots in available_slots_by_participant.items():
    print(f"\n[{participant_name}]")
    for slot in slots[:4]:
        print(f"  {slot.start_time.strftime('%Y-%m-%d %H:%M')} ~ {slot.end_time.strftime('%H:%M')}")

✓ 개인별 가용 시간대 계산 완료
  - 검색 기간: 2026-04-16 ~ 2026-04-20
    Alice:  61개 슬롯 (1830분)
    Bob:  62개 슬롯 (1860분)
    Charlie:  61개 슬롯 (1830분)
    Diana:  64개 슬롯 (1920분)
    Eve:  63개 슬롯 (1890분)

개인별 가용 시간대 (샘플 - 첫 2일):

[Alice]
  2026-04-16 09:00 ~ 10:00
  2026-04-16 09:30 ~ 10:30
  2026-04-16 10:00 ~ 11:00
  2026-04-16 10:30 ~ 11:30

[Bob]
  2026-04-16 09:00 ~ 10:00
  2026-04-16 09:30 ~ 10:30
  2026-04-16 10:00 ~ 11:00
  2026-04-16 10:30 ~ 11:30

[Charlie]
  2026-04-16 09:00 ~ 10:00
  2026-04-16 09:30 ~ 10:30
  2026-04-16 10:00 ~ 11:00
  2026-04-16 10:30 ~ 11:30

[Diana]
  2026-04-16 09:00 ~ 10:00
  2026-04-16 09:30 ~ 10:30
  2026-04-16 10:00 ~ 11:00
  2026-04-16 10:30 ~ 11:30

[Eve]
  2026-04-16 09:00 ~ 10:00
  2026-04-16 09:30 ~ 10:30
  2026-04-16 10:00 ~ 11:00
  2026-04-16 10:30 ~ 11:30


## Section 7: 전체 참가자 교집합 시간대 탐색

In [18]:
# 10. 전체 참가자 교집합 시간대 탐색
def find_common_slots(
    available_slots_by_participant: Dict[str, List[TimeSlot]],
    participants: List[Participant]
) -> List[TimeSlot]:
    """
    모든 참가자가 가능한 시간대 찾기 (교집합)
    """
    if not available_slots_by_participant:
        return []
    
    # 모든 슬롯을 시간대로 매핑
    all_slots_by_time = defaultdict(list)
    
    for participant_name, slots in available_slots_by_participant.items():
        for slot in slots:
            key = (slot.start_time, slot.end_time)
            if participant_name not in all_slots_by_time[key]:
                all_slots_by_time[key].append(participant_name)
    
    # 모든 참가자가 가능한 시간대만 필터링
    all_participant_names = {p.name for p in participants}
    common_slots = []
    
    for (start_time, end_time), participant_list in all_slots_by_time.items():
        if set(participant_list) == all_participant_names:
            common_slots.append(TimeSlot(
                start_time=start_time,
                end_time=end_time,
                available_participants=list(participant_list)
            ))
    
    # 시간순 정렬
    common_slots.sort(key=lambda s: s.start_time)
    return common_slots


# 교집합 계산
common_slots = find_common_slots(available_slots_by_participant, SAMPLE_PARTICIPANTS)
print(f"✓ 교집합 시간대 탐색 완료")
print(f"  - 모든 참가자가 가능한 시간대: {len(common_slots)}개")

if common_slots:
    print("\n공통 가능 시간대 (상위 5개):")
    for slot in common_slots[:5]:
        print(f"  ✓ {slot.start_time.strftime('%Y-%m-%d %H:%M')} ~ {slot.end_time.strftime('%H:%M')}")
        print(f"     참가자: {', '.join(slot.available_participants)}")
else:
    print("\n⚠️ 모든 참가자가 함께 가능한 시간대가 없습니다!")

# 부분 교집합 (일부 참가자만 필요한 경우)
def find_partial_slots(
    available_slots_by_participant: Dict[str, List[TimeSlot]],
    min_participants_count: int
) -> List[TimeSlot]:
    """
    최소 N명 이상이 가능한 시간대 찾기
    """
    all_slots_by_time = defaultdict(set)
    
    for participant_name, slots in available_slots_by_participant.items():
        for slot in slots:
            key = (slot.start_time, slot.end_time)
            all_slots_by_time[key].add(participant_name)
    
    partial_slots = []
    for (start_time, end_time), participant_set in all_slots_by_time.items():
        if len(participant_set) >= min_participants_count:
            partial_slots.append(TimeSlot(
                start_time=start_time,
                end_time=end_time,
                available_participants=list(participant_set)
            ))
    
    partial_slots.sort(key=lambda s: s.start_time)
    return partial_slots


# 부분 교집합 계산
for min_count in [5, 4, 3]:
    partial = find_partial_slots(available_slots_by_participant, min_count)
    if partial:
        print(f"\n{min_count}명 이상 가능한 시간: {len(partial)}개")
        print(f"  샘플: {partial[0].start_time.strftime('%Y-%m-%d %H:%M')} ~ {', '.join(partial[0].available_participants)}")
        break

✓ 교집합 시간대 탐색 완료
  - 모든 참가자가 가능한 시간대: 55개

공통 가능 시간대 (상위 5개):
  ✓ 2026-04-16 09:00 ~ 10:00
     참가자: Alice, Bob, Charlie, Diana, Eve
  ✓ 2026-04-16 09:30 ~ 10:30
     참가자: Alice, Bob, Charlie, Diana, Eve
  ✓ 2026-04-16 10:00 ~ 11:00
     참가자: Alice, Bob, Charlie, Diana, Eve
  ✓ 2026-04-16 10:30 ~ 11:30
     참가자: Alice, Bob, Charlie, Diana, Eve
  ✓ 2026-04-16 13:00 ~ 14:00
     참가자: Alice, Bob, Charlie, Diana, Eve

5명 이상 가능한 시간: 55개
  샘플: 2026-04-16 09:00 ~ Diana, Alice, Eve, Charlie, Bob


## Section 8: 공통 시간 부재 시 대체안 생성 (우선순위/제외 인원 반영)

In [19]:
# 11. 대체안 생성 (공통 시간이 없을 때)
def generate_alternative_recommendations(
    available_slots_by_participant: Dict[str, List[TimeSlot]],
    participants: List[Participant],
    required_count: int = None
) -> List[MeetingRecommendation]:
    """
    공통 시간이 없을 경우 대체안 생성
    - 필수 참석자는 반드시 포함
    - 우선순위 높은 사람부터 포함
    """
    if required_count is None:
        required_count = max(1, len(participants) - 1)  # 1명 빼도 됨
    
    # 우선순위 정렬
    sorted_participants = sorted(
        participants,
        key=lambda p: (p.priority == Priority.HIGH, p.priority == Priority.MEDIUM),
        reverse=True
    )
    
    # 각 참가자 조합별로 가능한 시간 찾기
    all_slots_by_time = defaultdict(set)
    for participant_name, slots in available_slots_by_participant.items():
        for slot in slots:
            key = (slot.start_time, slot.end_time)
            all_slots_by_time[key].add(participant_name)
    
    recommendations = []
    
    for (start_time, end_time), available_set in all_slots_by_time.items():
        available_count = len(available_set)
        
        if available_count >= required_count:
            # HIGH priority 참가자들을 우선적으로 포함
            high_priority = [p for p in sorted_participants if p.priority == Priority.HIGH]
            medium_priority = [p for p in sorted_participants if p.priority == Priority.MEDIUM]
            
            selected = []
            excluded = []
            
            # HIGH priority 우선 포함
            for p in high_priority:
                if p.name in available_set:
                    selected.append(p.name)
                else:
                    excluded.append(p.name)
            
            # 불족한 경우 MEDIUM priority 추가
            if len(selected) < required_count:
                for p in medium_priority:
                    if len(selected) >= required_count:
                        break
                    if p.name in available_set:
                        selected.append(p.name)
                    else:
                        excluded.append(p.name)
            
            if len(selected) >= required_count:
                confidence = available_count / len(participants)
                recommendations.append(MeetingRecommendation(
                    recommended_time=start_time,
                    available_participants=selected,
                    excluded_participants=excluded,
                    confidence_score=confidence,
                    reason=f"{len(selected)}명 가능 (제외: {', '.join(excluded) if excluded else '없음'})"
                ))
    
    # 신뢰도 점수 + 시간순으로 정렬
    recommendations.sort(
        key=lambda r: (-r.confidence_score, r.recommended_time)
    )
    
    return recommendations[:10]  # 상위 10개만 반환


# 대체안 생성
alternatives = generate_alternative_recommendations(
    available_slots_by_participant,
    SAMPLE_PARTICIPANTS
)

print(f"✓ 대체안 생성 완료")
print(f"  - 추천 시간대: {len(alternatives)}개")

if alternatives:
    print("\n추천 대체 시간대 (상위 3개):")
    for i, rec in enumerate(alternatives[:3], 1):
        print(f"\n{i}. {rec.recommended_time.strftime('%Y-%m-%d %H:%M')}")
        print(f"   참석: {', '.join(rec.available_participants)} ({len(rec.available_participants)}명)")
        if rec.excluded_participants:
            print(f"   제외: {', '.join(rec.excluded_participants)}")
        print(f"   신뢰도: {rec.confidence_score:.2%} | {rec.reason}")

✓ 대체안 생성 완료
  - 추천 시간대: 10개

추천 대체 시간대 (상위 3개):

1. 2026-04-16 09:00
   참석: Alice, Bob, Eve, Charlie (4명)
   신뢰도: 100.00% | 4명 가능 (제외: 없음)

2. 2026-04-16 09:30
   참석: Alice, Bob, Eve, Charlie (4명)
   신뢰도: 100.00% | 4명 가능 (제외: 없음)

3. 2026-04-16 10:00
   참석: Alice, Bob, Eve, Charlie (4명)
   신뢰도: 100.00% | 4명 가능 (제외: 없음)


## Section 9: 최적 회의 시간 스코어링 및 랭킹

In [20]:
# 12. 최적 회의 시간 스코어링
class ScoredRecommendation(BaseModel):
    """점수가 매겨진 추천"""
    recommendation: MeetingRecommendation
    base_score: float  # 신뢰도 (0-1)
    time_preference_score: float  # 시간 선호도 (0-1)
    priority_score: float  # 우선순위 반영 (0-1)
    total_score: float  # 최종 점수


def calculate_time_preference_score(meeting_time: datetime) -> float:
    """회의 시간대 선호도 점수 계산"""
    hour = meeting_time.hour
    if 10 <= hour < 12:
        return 1.0
    if 14 <= hour < 17:
        return 0.8
    if 9 <= hour < 18:
        return 0.6
    return 0.2


def calculate_priority_score(
    recommendation: MeetingRecommendation,
    all_participants: List[Participant],
) -> float:
    """참가자 우선순위 반영 점수 계산"""
    available_set = set(recommendation.available_participants)
    high_priority = [p for p in all_participants if p.priority == Priority.HIGH]
    medium_priority = [p for p in all_participants if p.priority == Priority.MEDIUM]

    high_available = sum(1 for p in high_priority if p.name in available_set)
    medium_available = sum(1 for p in medium_priority if p.name in available_set)

    high_ratio = high_available / max(1, len(high_priority))
    medium_ratio = medium_available / max(1, len(medium_priority))
    return high_ratio * 0.7 + medium_ratio * 0.3


def score_recommendations(
    recommendations: List[MeetingRecommendation],
    participants: List[Participant],
    weights: Optional[Dict[str, float]] = None,
) -> List[ScoredRecommendation]:
    """추천 목록을 점수화하고 랭킹 반환"""
    if weights is None:
        weights = {"base": 0.4, "time": 0.3, "priority": 0.3}

    scored: List[ScoredRecommendation] = []
    for rec in recommendations:
        base_score = rec.confidence_score
        time_score = calculate_time_preference_score(rec.recommended_time)
        priority_score = calculate_priority_score(rec, participants)
        total_score = (
            weights["base"] * base_score
            + weights["time"] * time_score
            + weights["priority"] * priority_score
        )

        scored.append(
            ScoredRecommendation(
                recommendation=rec,
                base_score=base_score,
                time_preference_score=time_score,
                priority_score=priority_score,
                total_score=total_score,
            )
        )

    scored.sort(key=lambda s: s.total_score, reverse=True)
    return scored


if alternatives:
    scored_recs = score_recommendations(alternatives, SAMPLE_PARTICIPANTS)
    print("✓ 최적 회의 시간 스코어링 완료")
    print("\n점수 계산 결과 (상위 3개):")
    print(f"{'순위':<4}{'시간':<20}{'신뢰도':<8}{'시간선호':<10}{'우선순위':<10}{'최종점수':<8}")
    print("-" * 68)

    for i, scored in enumerate(scored_recs[:3], 1):
        print(
            f"{i:<4}"
            f"{scored.recommendation.recommended_time.strftime('%Y-%m-%d %H:%M'):<20}"
            f"{scored.base_score:<8.2f}"
            f"{scored.time_preference_score:<10.2f}"
            f"{scored.priority_score:<10.2f}"
            f"{scored.total_score:<8.3f}"
        )
        print(f"     -> {', '.join(scored.recommendation.available_participants)}")

✓ 최적 회의 시간 스코어링 완료

점수 계산 결과 (상위 3개):
순위  시간                  신뢰도     시간선호      우선순위      최종점수    
--------------------------------------------------------------------
1   2026-04-16 10:00    1.00    1.00      0.85      0.955   
     -> Alice, Bob, Eve, Charlie
2   2026-04-16 10:30    1.00    1.00      0.85      0.955   
     -> Alice, Bob, Eve, Charlie
3   2026-04-16 14:00    1.00    0.80      0.85      0.895   
     -> Alice, Bob, Eve, Charlie


## Section 10: 조건부 재시도/폴백 전략 구현

In [21]:
# 13. 조건부 재시도/폴백 전략
import time as pytime


class RetryConfig(BaseModel):
    """재시도 설정"""
    max_retries: int = 3
    initial_delay_seconds: float = 0.1
    backoff_factor: float = 2.0
    max_delay_seconds: float = 10.0


class CircuitBreaker:
    """회로차단기 패턴"""

    def __init__(self, failure_threshold: int = 5, timeout_seconds: float = 60):
        self.failure_threshold = failure_threshold
        self.timeout_seconds = timeout_seconds
        self.failure_count = 0
        self.last_failure_time: Optional[float] = None
        self.state = "CLOSED"  # CLOSED, OPEN, HALF_OPEN

    def is_open(self) -> bool:
        if self.state == "OPEN":
            if self.last_failure_time is not None and (pytime.time() - self.last_failure_time > self.timeout_seconds):
                self.state = "HALF_OPEN"
                return False
            return True
        return False

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = pytime.time()
        if self.failure_count >= self.failure_threshold:
            self.state = "OPEN"


def retry_with_exponential_backoff(
    func,
    fallback_func,
    config: Optional[RetryConfig] = None,
    circuit_breaker: Optional[CircuitBreaker] = None,
    *args,
    **kwargs,
) -> dict:
    """지수 백오프 재시도 후 실패 시 폴백 실행"""
    if config is None:
        config = RetryConfig()
    if circuit_breaker is None:
        circuit_breaker = CircuitBreaker()

    if circuit_breaker.is_open():
        logger.warning("Circuit breaker OPEN: using fallback immediately")
        try:
            result = fallback_func(*args, **kwargs)
            result["fallback_reason"] = "circuit_breaker_open"
            return result
        except Exception as e:
            circuit_breaker.record_failure()
            return {"status": "failed", "error": str(e), "source": "fallback"}

    delay = config.initial_delay_seconds
    last_error = None

    for attempt in range(config.max_retries):
        try:
            result = func(*args, **kwargs)
            circuit_breaker.record_success()
            result["attempt"] = attempt + 1
            result["status"] = "success"
            logger.info(f"✓ 성공 (시도 {attempt + 1}/{config.max_retries})")
            return result
        except Exception as e:
            last_error = e
            circuit_breaker.record_failure()
            logger.warning(f"✗ 시도 {attempt + 1} 실패: {str(e)}")
            if attempt < config.max_retries - 1:
                logger.info(f"  -> {delay:.2f}초 후 재시도...")
                pytime.sleep(delay)
                delay = min(delay * config.backoff_factor, config.max_delay_seconds)

    logger.warning("✗ 모든 재시도 실패. 폴백 함수 사용")
    try:
        result = fallback_func(*args, **kwargs)
        result["fallback_reason"] = "all_retries_failed"
        result["original_error"] = str(last_error)
        return result
    except Exception as fallback_error:
        circuit_breaker.record_failure()
        return {
            "status": "failed",
            "error": str(last_error),
            "fallback_error": str(fallback_error),
            "source": "both_failed",
        }


failure_count = 0


def unstable_api_call(participant_name: str) -> dict:
    """불안정한 API (처음 2회 실패)"""
    global failure_count
    failure_count += 1
    if failure_count <= 2:
        raise Exception(f"API 일시적 오류 (시도 {failure_count})")
    return {"status": "success", "data": f"{participant_name} 일정 데이터"}


def fallback_api_call(participant_name: str) -> dict:
    """폴백: 캐시 데이터 사용"""
    return {"status": "fallback", "data": f"{participant_name} 캐시된 일정 (최근 24시간)"}


print("✓ 재시도/폴백 전략 테스트")
print("\n[테스트 1] 재시도 후 성공")
failure_count = 0
result = retry_with_exponential_backoff(
    unstable_api_call,
    fallback_api_call,
    participant_name="Alice",
)
print(f"결과: {result}\n")


def always_fail(participant_name: str) -> dict:
    raise Exception("영구적 API 오류")


print("[테스트 2] 완전 실패 후 폴백")
result = retry_with_exponential_backoff(
    always_fail,
    fallback_api_call,
    config=RetryConfig(max_retries=2),
    participant_name="Bob",
)
print(f"결과: {result}")

INFO:__main__:  -> 0.10초 후 재시도...


✓ 재시도/폴백 전략 테스트

[테스트 1] 재시도 후 성공


INFO:__main__:  -> 0.20초 후 재시도...
INFO:__main__:✓ 성공 (시도 3/3)
INFO:__main__:  -> 0.10초 후 재시도...


결과: {'status': 'success', 'data': 'Alice 일정 데이터', 'attempt': 3}

[테스트 2] 완전 실패 후 폴백
결과: {'status': 'fallback', 'data': 'Bob 캐시된 일정 (최근 24시간)', 'fallback_reason': 'all_retries_failed', 'original_error': '영구적 API 오류'}


## Section 11: 상태 체크포인트 저장/복구 (DB/파일 기반)

In [22]:
# 14. 체크포인트 시스템 (상태 저장/복구)
import hashlib
from pathlib import Path
from typing import Any


class Checkpoint(BaseModel):
    """체크포인트 저장 단위"""
    step_name: str
    timestamp: datetime
    version: int = 1
    data: Dict[str, Any]
    checksum: str = ""

    def calculate_checksum(self) -> str:
        data_str = json.dumps(self.data, default=str, sort_keys=True)
        return hashlib.md5(data_str.encode("utf-8")).hexdigest()


class CheckpointManager:
    """체크포인트 관리자"""

    def __init__(self, checkpoint_dir: str = ".checkpoints"):
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(exist_ok=True)
        self.checkpoints: Dict[str, Checkpoint] = {}

    def save_checkpoint(self, step_name: str, data: Dict[str, Any]) -> Checkpoint:
        checkpoint = Checkpoint(
            step_name=step_name,
            timestamp=datetime.now(KST),
            data=data,
        )
        checkpoint.checksum = checkpoint.calculate_checksum()

        self.checkpoints[step_name] = checkpoint

        checkpoint_file = self.checkpoint_dir / f"{step_name}.json"
        with open(checkpoint_file, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "step_name": checkpoint.step_name,
                    "timestamp": checkpoint.timestamp.isoformat(),
                    "version": checkpoint.version,
                    "checksum": checkpoint.checksum,
                    "data": checkpoint.data,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        logger.info(f"✓ 체크포인트 저장: {step_name} (checksum: {checkpoint.checksum[:8]}...)")
        return checkpoint

    def load_checkpoint(self, step_name: str) -> Optional[Checkpoint]:
        if step_name in self.checkpoints:
            return self.checkpoints[step_name]

        checkpoint_file = self.checkpoint_dir / f"{step_name}.json"
        if not checkpoint_file.exists():
            return None

        with open(checkpoint_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        checkpoint = Checkpoint(
            step_name=data["step_name"],
            timestamp=datetime.fromisoformat(data["timestamp"]),
            version=data.get("version", 1),
            data=data["data"],
            checksum=data["checksum"],
        )

        expected = checkpoint.calculate_checksum()
        if checkpoint.checksum != expected:
            logger.warning(f"⚠ 체크포인트 무결성 오류: {step_name}")
            return None

        self.checkpoints[step_name] = checkpoint
        logger.info(f"✓ 체크포인트 로드: {step_name}")
        return checkpoint

    def get_last_successful_step(self) -> Optional[str]:
        if not self.checkpoints:
            return None
        latest = max(self.checkpoints.values(), key=lambda c: c.timestamp)
        return latest.step_name

    def list_checkpoints(self) -> List[Dict[str, Any]]:
        checkpoints_list: List[Dict[str, Any]] = []
        for file in self.checkpoint_dir.glob("*.json"):
            cp = self.load_checkpoint(file.stem)
            if cp:
                checkpoints_list.append(
                    {
                        "step": cp.step_name,
                        "timestamp": cp.timestamp,
                        "version": cp.version,
                    }
                )
        return sorted(checkpoints_list, key=lambda x: x["timestamp"])

    def clear_checkpoints(self):
        for file in self.checkpoint_dir.glob("*.json"):
            file.unlink()
        self.checkpoints.clear()
        logger.info("✓ 모든 체크포인트 삭제")


checkpoint_manager = CheckpointManager()
print("✓ 체크포인트 시스템 테스트")

steps_data = {
    "step_1_input": {
        "participants": [p.model_dump() for p in SAMPLE_PARTICIPANTS[:2]],
        "duration_minutes": 60,
    },
    "step_2_collect": {"total_events": 12, "sources": ["Teams", "Internal DB"]},
    "step_3_standardize": {"standardized_count": 12, "duplicates_removed": 2},
}

for step_name, data in steps_data.items():
    checkpoint_manager.save_checkpoint(step_name, data)

print("\n저장된 체크포인트:")
for cp in checkpoint_manager.list_checkpoints():
    print(f"  - {cp['step']}: {cp['timestamp'].strftime('%H:%M:%S')}")

print(f"\n마지막 성공 단계: {checkpoint_manager.get_last_successful_step()}")
recovered = checkpoint_manager.load_checkpoint("step_2_collect")
if recovered:
    print("\n복구된 데이터:")
    print(f"  {recovered.data}")

INFO:__main__:✓ 체크포인트 저장: step_1_input (checksum: 113ad33a...)
INFO:__main__:✓ 체크포인트 저장: step_2_collect (checksum: 8b789991...)
INFO:__main__:✓ 체크포인트 저장: step_3_standardize (checksum: 2cb71477...)


✓ 체크포인트 시스템 테스트

저장된 체크포인트:
  - step_1_input: 13:34:46
  - step_2_collect: 13:34:46
  - step_3_standardize: 13:34:46

마지막 성공 단계: step_3_standardize

복구된 데이터:
  {'total_events': 12, 'sources': ['Teams', 'Internal DB']}


## Section 12: 일정 확정, 알림 발송, 캘린더 초대 자동화

In [23]:
# 15. 일정 확정 및 알림 발송
class CalendarInvitation(BaseModel):
    """캘린더 초대"""
    meeting_time: datetime
    organizer: str
    participants: List[str]
    excluded_participants: List[str] = []
    title: str
    description: str
    meeting_link: str
    confidence_score: float
    is_teams: bool = True


def generate_calendar_invitation(
    recommendation: MeetingRecommendation,
    organizer: str = "HR",
    title: str = "팀 전체 회의",
) -> CalendarInvitation:
    """캘린더 초대 생성"""
    start_time = recommendation.recommended_time
    meeting_link = f"https://teams.microsoft.com/l/meetup-join/{int(start_time.timestamp())}"

    description = (
        "자동 조율된 회의입니다.\n"
        f"신뢰도: {recommendation.confidence_score:.0%}\n"
        f"사유: {recommendation.reason}"
    )

    return CalendarInvitation(
        meeting_time=start_time,
        organizer=organizer,
        participants=recommendation.available_participants,
        excluded_participants=recommendation.excluded_participants,
        title=title,
        description=description,
        meeting_link=meeting_link,
        confidence_score=recommendation.confidence_score,
        is_teams=True,
    )


class NotificationPayload(BaseModel):
    """알림 페이로드"""
    channel: str
    recipient_id: str
    title: str
    message: str
    action_url: str = ""
    priority: str = "normal"
    timestamp: datetime = Field(default_factory=lambda: datetime.now(KST))


def generate_notification_payloads(
    invitation: CalendarInvitation,
    channels: List[str] = ["teams", "email"],
) -> List[NotificationPayload]:
    """채널별 알림 페이로드 생성"""
    payloads: List[NotificationPayload] = []
    meeting_time_str = invitation.meeting_time.strftime("%Y-%m-%d %H:%M")
    participants_str = ", ".join(invitation.participants)

    for channel in channels:
        if channel == "teams":
            payload = NotificationPayload(
                channel="teams",
                recipient_id="team_channel",
                title=f"[일정 확정] {invitation.title}",
                message=(
                    "회의가 자동으로 조율되었습니다.\n\n"
                    f"시간: {meeting_time_str}\n"
                    f"참석자: {participants_str}\n"
                    f"링크: {invitation.meeting_link}\n"
                    f"신뢰도: {invitation.confidence_score:.0%}"
                ),
                action_url=invitation.meeting_link,
                priority="high",
            )
        elif channel == "email":
            payload = NotificationPayload(
                channel="email",
                recipient_id="team@company.com",
                title=f"[회의 초대] {invitation.title}",
                message=(
                    "안녕하세요,\n\n"
                    "회의 일정이 자동으로 결정되었습니다.\n\n"
                    f"날짜/시간: {meeting_time_str}\n"
                    f"참석자: {participants_str}\n"
                    "캘린더를 확인해주세요."
                ),
                priority="normal",
            )
        elif channel == "messenger":
            payload = NotificationPayload(
                channel="messenger",
                recipient_id="internal_team_channel",
                title="회의 일정 안내",
                message=f"{meeting_time_str} 회의 참석 요청: {participants_str}",
                action_url=invitation.meeting_link,
                priority="normal",
            )
        else:
            continue

        payloads.append(payload)

    return payloads


class MockNotificationSender:
    """알림 발송 시뮬레이터"""

    def __init__(self):
        self.sent_notifications: List[NotificationPayload] = []

    def send(self, payload: NotificationPayload) -> dict:
        self.sent_notifications.append(payload)
        logger.info(f"[{payload.channel.upper()}] {payload.recipient_id}: {payload.title}")
        return {
            "status": "sent",
            "channel": payload.channel,
            "recipient": payload.recipient_id,
            "timestamp": datetime.now(KST).isoformat(),
        }

    def send_batch(self, payloads: List[NotificationPayload]) -> dict:
        results = [self.send(payload) for payload in payloads]
        return {"status": "completed", "total_sent": len(results), "results": results}


if alternatives:
    print("✓ 알림 발송 및 캘린더 초대 자동화")
    best_recommendation = alternatives[0]
    invitation = generate_calendar_invitation(
        best_recommendation,
        organizer="HR Manager",
        title="분기별 전략 수립 회의",
    )

    print("\n캘린더 초대:")
    print(f"  시간: {invitation.meeting_time.strftime('%Y-%m-%d %H:%M')}")
    print(f"  참석자: {', '.join(invitation.participants)}")
    if invitation.excluded_participants:
        print(f"  불참: {', '.join(invitation.excluded_participants)}")
    print(f"  링크: {invitation.meeting_link}")

    payloads = generate_notification_payloads(
        invitation,
        channels=["teams", "email", "messenger"],
    )
    print(f"\n생성된 알림 페이로드: {len(payloads)}개")

    sender = MockNotificationSender()
    send_result = sender.send_batch(payloads)
    print("\n발송 결과:")
    print(f"  상태: {send_result['status']}")
    print(f"  총 발송: {send_result['total_sent']}개")

INFO:__main__:[TEAMS] team_channel: [일정 확정] 분기별 전략 수립 회의
INFO:__main__:[EMAIL] team@company.com: [회의 초대] 분기별 전략 수립 회의
INFO:__main__:[MESSENGER] internal_team_channel: 회의 일정 안내


✓ 알림 발송 및 캘린더 초대 자동화

캘린더 초대:
  시간: 2026-04-16 09:00
  참석자: Alice, Bob, Eve, Charlie
  링크: https://teams.microsoft.com/l/meetup-join/1776297600

생성된 알림 페이로드: 3개

발송 결과:
  상태: completed
  총 발송: 3개


## Section 13: 노드 기반 워크플로우 오케스트레이션 (LangGraph)

In [24]:
# 16. LangGraph 기반 워크플로우
import time as pytime


class WorkflowState(BaseModel):
    """워크플로우 상태"""
    participants: List[Participant]
    calendar_data: Dict[str, List[CalendarEvent]]
    available_slots: List[TimeSlot]
    recommendation: Optional[MeetingRecommendation]
    invitation: Optional[CalendarInvitation]
    status: str
    errors: List[str] = []
    executed_steps: List[str] = []


class WorkflowExecutionLog(BaseModel):
    """워크플로우 실행 로그"""
    step_name: str
    status: str
    start_time: datetime
    end_time: datetime
    duration_seconds: float
    error_message: Optional[str] = None
    retry_count: int = 0


class MeetingSchedulerWorkflow:
    """회의 일정 조율 워크플로우"""

    def __init__(self, participants: List[Participant]):
        self.participants = participants
        self.checkpoint_manager = CheckpointManager()
        self.execution_logs: List[WorkflowExecutionLog] = []
        self.notification_sender = MockNotificationSender()

    def log_step(
        self,
        step_name: str,
        status: str,
        duration: float,
        error: Optional[str] = None,
        retry_count: int = 0,
    ):
        log = WorkflowExecutionLog(
            step_name=step_name,
            status=status,
            start_time=datetime.now(KST),
            end_time=datetime.now(KST),
            duration_seconds=duration,
            error_message=error,
            retry_count=retry_count,
        )
        self.execution_logs.append(log)

    def node_1_input(self, state: dict) -> dict:
        start = pytime.time()
        try:
            state["participants"] = self.participants
            state["status"] = "input_complete"
            state.setdefault("executed_steps", []).append("node_1_input")
            self.log_step("node_1_input", "success", pytime.time() - start)
            return state
        except Exception as e:
            self.log_step("node_1_input", "failed", pytime.time() - start, str(e))
            raise

    def node_2_collect_calendar(self, state: dict) -> dict:
        start = pytime.time()
        try:
            calendar_data = {}
            for participant in state.get("participants", self.participants):
                events = fetch_calendar_from_multiple_sources(participant, list(connectors.values()))
                calendar_data[participant.name] = events

            state["calendar_data"] = calendar_data
            state.setdefault("executed_steps", []).append("node_2_collect_calendar")
            self.checkpoint_manager.save_checkpoint(
                "node_2_collect",
                {"count": sum(len(e) for e in calendar_data.values())},
            )
            self.log_step("node_2_collect", "success", pytime.time() - start)
            return state
        except Exception as e:
            self.log_step("node_2_collect", "failed", pytime.time() - start, str(e))
            state.setdefault("errors", []).append(f"수집 실패: {str(e)}")
            return state

    def node_3_standardize(self, state: dict) -> dict:
        start = pytime.time()
        try:
            standardized = standardize_calendar_data(state["calendar_data"])
            normalized = normalize_timezone(standardized)
            state["calendar_data"] = normalized
            state.setdefault("executed_steps", []).append("node_3_standardize")
            self.checkpoint_manager.save_checkpoint("node_3_standardize", {"status": "completed"})
            self.log_step("node_3_standardize", "success", pytime.time() - start)
            return state
        except Exception as e:
            self.log_step("node_3_standardize", "failed", pytime.time() - start, str(e))
            state.setdefault("errors", []).append(f"표준화 실패: {str(e)}")
            return state

    def node_4_identify_fixed(self, state: dict) -> dict:
        start = pytime.time()
        try:
            fixed_schedules = identify_fixed_schedules_rule_based(state["calendar_data"])
            state["fixed_schedules"] = fixed_schedules
            state.setdefault("executed_steps", []).append("node_4_identify_fixed")
            self.checkpoint_manager.save_checkpoint("node_4_fixed", {"fixed_count": len(fixed_schedules)})
            self.log_step("node_4_fixed", "success", pytime.time() - start)
            return state
        except Exception as e:
            self.log_step("node_4_fixed", "failed", pytime.time() - start, str(e))
            state.setdefault("errors", []).append(f"불변일정 식별 실패: {str(e)}")
            return state

    def node_5_find_common_slots(self, state: dict) -> dict:
        start = pytime.time()
        try:
            today = datetime.now(KST).replace(hour=0, minute=0, second=0, microsecond=0)
            date_range = (today, today + timedelta(days=4))

            available_by_participant = {}
            for name, events in state["calendar_data"].items():
                slots = calculate_available_slots(name, events, date_range)
                available_by_participant[name] = slots

            common_slots = find_common_slots(
                available_by_participant,
                state.get("participants", self.participants),
            )
            state["available_slots"] = common_slots
            state["has_common_slot"] = len(common_slots) > 0
            state.setdefault("executed_steps", []).append("node_5_find_common_slots")
            self.checkpoint_manager.save_checkpoint("node_5_common", {"common_count": len(common_slots)})
            self.log_step("node_5_common", "success", pytime.time() - start)
            return state
        except Exception as e:
            self.log_step("node_5_common", "failed", pytime.time() - start, str(e))
            state.setdefault("errors", []).append(f"교집합 탐색 실패: {str(e)}")
            return state

    def node_6_check_common_time(self, state: dict) -> str:
        return "confirm_meeting" if state.get("has_common_slot", False) else "suggest_alternative"

    def node_7_suggest_alternative(self, state: dict) -> dict:
        start = pytime.time()
        try:
            today = datetime.now(KST).replace(hour=0, minute=0, second=0, microsecond=0)
            date_range = (today, today + timedelta(days=4))

            available_by_participant = {}
            for name, events in state["calendar_data"].items():
                slots = calculate_available_slots(name, events, date_range)
                available_by_participant[name] = slots

            recs = generate_alternative_recommendations(
                available_by_participant,
                state.get("participants", self.participants),
            )
            if recs:
                state["recommendation"] = recs[0]
                state["alternatives"] = recs[:3]

            state.setdefault("executed_steps", []).append("node_7_suggest_alternative")
            self.log_step("node_7_alternative", "success", pytime.time() - start)
            return state
        except Exception as e:
            self.log_step("node_7_alternative", "failed", pytime.time() - start, str(e))
            state.setdefault("errors", []).append(f"대체안 생성 실패: {str(e)}")
            return state

    def node_8_confirm_meeting(self, state: dict) -> dict:
        start = pytime.time()
        try:
            if not state.get("recommendation") and state.get("available_slots"):
                slot = state["available_slots"][0]
                state["recommendation"] = MeetingRecommendation(
                    recommended_time=slot.start_time,
                    available_participants=slot.available_participants,
                    confidence_score=1.0,
                    reason="모든 참가자 가능",
                )

            if state.get("recommendation"):
                invitation = generate_calendar_invitation(
                    state["recommendation"],
                    title="자동 조율 회의",
                )
                state["invitation"] = invitation
                payloads = generate_notification_payloads(invitation, channels=["teams", "email"])
                self.notification_sender.send_batch(payloads)
                state["status"] = "meeting_confirmed"

            state.setdefault("executed_steps", []).append("node_8_confirm_meeting")
            self.checkpoint_manager.save_checkpoint("node_8_confirm", {"status": state.get("status", "unknown")})
            self.log_step("node_8_confirm", "success", pytime.time() - start)
            return state
        except Exception as e:
            self.log_step("node_8_confirm", "failed", pytime.time() - start, str(e))
            state.setdefault("errors", []).append(f"확정 실패: {str(e)}")
            return state

    def node_9_complete(self, state: dict) -> dict:
        state["status"] = "workflow_complete"
        state.setdefault("executed_steps", []).append("node_9_complete")
        return state


print("✓ LangGraph 워크플로우 구성 완료")
print("  - 총 9개 노드")
print("  - 1개 분기점 (node_6)")

✓ LangGraph 워크플로우 구성 완료
  - 총 9개 노드
  - 1개 분기점 (node_6)


## Section 14: 테스트 시나리오 (정상/충돌/API 실패) 및 성능 측정

In [25]:
# 17. 통합 테스트 및 성능 측정
import time as pytime

print("=" * 60)
print("MeetingScheduler 워크플로우 통합 테스트")
print("=" * 60)

print("\n[테스트 1] 정상 케이스")
print("-" * 60)

workflow = MeetingSchedulerWorkflow(SAMPLE_PARTICIPANTS)
state = {
    "participants": SAMPLE_PARTICIPANTS,
    "calendar_data": {},
    "available_slots": [],
    "recommendation": None,
    "invitation": None,
    "status": "init",
    "errors": [],
    "executed_steps": [],
    "has_common_slot": False,
    "fixed_schedules": [],
}

start_time = pytime.time()
state = workflow.node_1_input(state)
state = workflow.node_2_collect_calendar(state)
state = workflow.node_3_standardize(state)
state = workflow.node_4_identify_fixed(state)
state = workflow.node_5_find_common_slots(state)

if workflow.node_6_check_common_time(state) == "confirm_meeting":
    state = workflow.node_8_confirm_meeting(state)
    print("✓ 공통 가능 시간 발견")
else:
    state = workflow.node_7_suggest_alternative(state)
    state = workflow.node_8_confirm_meeting(state)
    print("⚠ 공통 시간 없음 - 대체안 제시")

state = workflow.node_9_complete(state)
total_time = pytime.time() - start_time

print("\n✓ 워크플로우 완료")
print(f"  - 상태: {state['status']}")
print(f"  - 처리 시간: {total_time:.3f}초")
if state.get("invitation"):
    print(f"  - 확정 회의: {state['invitation'].meeting_time.strftime('%Y-%m-%d %H:%M')}")
    print(f"  - 참석자: {', '.join(state['invitation'].participants)}")

if state["errors"]:
    print("\n⚠ 발생한 오류:")
    for error in state["errors"]:
        print(f"  - {error}")

print("\n실행 로그:")
print(f"{'단계':<30}{'상태':<12}{'소요시간':<10}")
print("-" * 56)
for log in workflow.execution_logs:
    print(f"{log.step_name:<30}{log.status:<12}{log.duration_seconds:.4f}s")

total_duration = sum(log.duration_seconds for log in workflow.execution_logs)
print("-" * 56)
print(f"{'총 소요 시간':<30}{'':<12}{total_duration:.4f}s")

print("\n성능 분석:")
print(f"  - 총 참가자: {len(SAMPLE_PARTICIPANTS)}명")
print(f"  - 총 일정: {sum(len(events) for events in state.get('calendar_data', {}).values())}건")
print(f"  - 처리 속도: {len(SAMPLE_PARTICIPANTS) / (total_time + 1e-3):.1f} 명/초")

print("\n저장된 체크포인트:")
for cp in workflow.checkpoint_manager.list_checkpoints():
    print(f"  - {cp['step']}: {cp['timestamp'].strftime('%H:%M:%S')}")

print("\n" + "=" * 60)
print("테스트 완료")
print("=" * 60)

INFO:__main__:[Teams] 수집 중: Alice
INFO:__main__:[Internal DB] 수집 중: Alice
INFO:__main__:[Messenger] 수집 중: Alice
INFO:__main__:[Teams] 수집 중: Bob
INFO:__main__:[Internal DB] 수집 중: Bob
INFO:__main__:[Messenger] 수집 중: Bob
INFO:__main__:[Teams] 수집 중: Charlie
INFO:__main__:[Internal DB] 수집 중: Charlie
INFO:__main__:[Messenger] 수집 중: Charlie
INFO:__main__:[Teams] 수집 중: Diana
INFO:__main__:[Internal DB] 수집 중: Diana
INFO:__main__:[Messenger] 수집 중: Diana
INFO:__main__:[Teams] 수집 중: Eve
INFO:__main__:[Internal DB] 수집 중: Eve
INFO:__main__:[Messenger] 수집 중: Eve
INFO:__main__:✓ 체크포인트 저장: node_2_collect (checksum: b2bad5f2...)
INFO:__main__:✓ 체크포인트 저장: node_3_standardize (checksum: 33d93cb6...)
INFO:__main__:✓ 체크포인트 저장: node_4_fixed (checksum: fbf17fbb...)
INFO:__main__:✓ 체크포인트 저장: node_5_common (checksum: f011f806...)
INFO:__main__:[TEAMS] team_channel: [일정 확정] 자동 조율 회의
INFO:__main__:[EMAIL] team@company.com: [회의 초대] 자동 조율 회의
INFO:__main__:✓ 체크포인트 저장: node_8_confirm (checksum: befc104e...)
INFO:__mai

MeetingScheduler 워크플로우 통합 테스트

[테스트 1] 정상 케이스
------------------------------------------------------------
✓ 공통 가능 시간 발견

✓ 워크플로우 완료
  - 상태: workflow_complete
  - 처리 시간: 0.037초
  - 확정 회의: 2026-04-16 09:00
  - 참석자: Alice, Bob, Charlie, Diana, Eve

실행 로그:
단계                            상태          소요시간      
--------------------------------------------------------
node_1_input                  success     0.0000s
node_2_collect                success     0.0162s
node_3_standardize            success     0.0026s
node_4_fixed                  success     0.0041s
node_5_common                 success     0.0066s
node_8_confirm                success     0.0053s
--------------------------------------------------------
총 소요 시간                                   0.0347s

성능 분석:
  - 총 참가자: 5명
  - 총 일정: 6건
  - 처리 속도: 132.5 명/초

저장된 체크포인트:
  - step_1_input: 13:34:46
  - step_2_collect: 13:34:46
  - step_3_standardize: 13:34:46
  - node_2_collect: 13:34:47
  - node_3_standardize: 13:34:47
  - node_4

## 최종 요약 및 사용 가이드

In [26]:
# 최종 요약
print("""
============================================================
MeetingScheduler: 회의 일정 자동 조율 Agent
============================================================

구현된 핵심 기능
- 다중 소스 일정 수집 (Teams, 내부망, 메신저)
- 일정 표준화 및 타임존 정규화
- 불변 일정 식별
- 개인별 가용 시간 계산 및 교집합 탐색
- 공통 시간 부재 시 대체 시간 추천
- 우선순위/시간 선호 기반 스코어링
- 체크포인트 기반 복구
- 자동 알림/캘린더 초대 생성

실행 순서
1) Section 1~8 실행 (데이터 준비)
2) Section 9~14 실행 (스코어링~워크플로우)
3) Section 14 테스트 셀 실행 후 결과 확인

문제 해결 팁
- 패키지 오류: kernel에 pandas/langgraph 설치 확인
- API 실패: RetryConfig 재시도 값 조정
- 체크포인트 초기화: CheckpointManager().clear_checkpoints()

운영 적용 시
- 더미 커넥터를 실제 API 커넥터로 교체
- 알림 수신자를 실제 조직 채널/메일로 변경
- 업무시간, 슬롯 크기, 우선순위 가중치 튜닝
============================================================
""")


MeetingScheduler: 회의 일정 자동 조율 Agent

구현된 핵심 기능
- 다중 소스 일정 수집 (Teams, 내부망, 메신저)
- 일정 표준화 및 타임존 정규화
- 불변 일정 식별
- 개인별 가용 시간 계산 및 교집합 탐색
- 공통 시간 부재 시 대체 시간 추천
- 우선순위/시간 선호 기반 스코어링
- 체크포인트 기반 복구
- 자동 알림/캘린더 초대 생성

실행 순서
1) Section 1~8 실행 (데이터 준비)
2) Section 9~14 실행 (스코어링~워크플로우)
3) Section 14 테스트 셀 실행 후 결과 확인

문제 해결 팁
- 패키지 오류: kernel에 pandas/langgraph 설치 확인
- API 실패: RetryConfig 재시도 값 조정
- 체크포인트 초기화: CheckpointManager().clear_checkpoints()

운영 적용 시
- 더미 커넥터를 실제 API 커넥터로 교체
- 알림 수신자를 실제 조직 채널/메일로 변경
- 업무시간, 슬롯 크기, 우선순위 가중치 튜닝

